In [1]:
# %pip install aeon seaborn torch numpy pandas

In [2]:
import sys
import random
import numpy as np
import time
import gc
import warnings
from sklearn.metrics import accuracy_score

warnings.filterwarnings("ignore")

In [3]:
from aeon.classification.distance_based import KNeighborsTimeSeriesClassifier as DTWClassifier
from aeon.classification.convolution_based import RocketClassifier, MultiRocketHydraClassifier, HydraClassifier
from aeon.classification.hybrid import HIVECOTEV2
from aeon.datasets import load_classification

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


In [4]:
UEA_MTSC30 = ['EthanolConcentration',
              'FaceDetection',
              'Handwriting',
              'Heartbeat',
              'JapaneseVowels',
              'PEMS-SF',
              'SelfRegulationSCP1',
              'SelfRegulationSCP2',
              'SpokenArabicDigits',
              'UWaveGestureLibrary',
              'ArticularyWordRecognition',
              'AtrialFibrillation',
              'BasicMotions',
              'CharacterTrajectories',
              'Cricket',
              'DuckDuckGeese',
              'Epilepsy',
              'ERing',
              'FingerMovements',
              'HandMovementDirection',
              'InsectWingbeat',
              'Libras',
              'LSST',
              'MotorImagery',
              'NATOPS',
              'PenDigits',
              'PhonemeSpectra',
              'RacketSports',
              'StandWalkJump',
              'EigenWorms',]

SEED = 0

In [5]:
def timeit(func):
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        print(f"\tFunction {func.__name__} took {end_time - start_time:.4f} seconds to execute.")
        return result
    return wrapper

In [6]:
@timeit
def train_model(classifier, X_train, y_train):
    classifier.fit(X_train, y_train)
    return classifier

@timeit
def eval_model(classifier, X_test, y_test):
    y_pred = classifier.predict(X_test)
    return accuracy_score(y_test, y_pred)


In [7]:
models = {
    'DTW': DTWClassifier(n_jobs=8),
    'Rocket': RocketClassifier(random_state=SEED, n_jobs=8),
    'Hydra': HydraClassifier(random_state=SEED, n_jobs=8),
    'Hive-Cote v2': HIVECOTEV2(time_limit_in_minutes=0.2, random_state=SEED, n_jobs=8),
    'MultiRocket-Hydra': MultiRocketHydraClassifier(random_state=SEED, n_jobs=8),
}
for dname in UEA_MTSC30:
    print(f"===== {dname} =====")
    X_train, y_train = load_classification(dname, split="train", extract_path="./Multivariate_ts", load_equal_length=True)
    X_test, y_test = load_classification(dname, split="test", extract_path="./Multivariate_ts", load_equal_length=True)
    print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
    for model in models:
        print(f"\t===== {model} =====")
        random.seed(SEED)
        np.random.seed(SEED)
        models[model] = train_model(models[model], X_train, y_train)
        accuracy = eval_model(models[model], X_test, y_test)
        print(f"\t{model} on {dname}: {accuracy * 100} %")
        print(f"\t{model} on {dname} object size is {sys.getsizeof(models[model])/1024} KB")
        models[model].reset(keep=None)
        gc.collect()

===== EthanolConcentration =====
Train shape: (261, 3, 1751), Test shape: (263, 3, 1751)
	===== DTW =====
	Function train_model took 0.0054 seconds to execute.
	Function eval_model took 1160.1139 seconds to execute.
	DTW on EthanolConcentration: 32.31939163498099 %
	DTW on EthanolConcentration object size is 0.046875 KB
	===== Rocket =====
	Function train_model took 9.6479 seconds to execute.
	Function eval_model took 9.7186 seconds to execute.
	Rocket on EthanolConcentration: 43.346007604562736 %
	Rocket on EthanolConcentration object size is 0.046875 KB
	===== Hydra =====
	Function train_model took 12.2907 seconds to execute.
	Function eval_model took 12.7587 seconds to execute.
	Hydra on EthanolConcentration: 54.372623574144484 %
	Hydra on EthanolConcentration object size is 0.046875 KB
	===== Hive-Cote v2 =====
	Function train_model took 68.4112 seconds to execute.
	Function eval_model took 63.3950 seconds to execute.
	Hive-Cote v2 on EthanolConcentration: 47.14828897338403 %
	Hive

In [8]:
models = {
    'MultiRocket-Hydra': MultiRocketHydraClassifier(random_state=SEED, n_jobs=8),
}
for dname in ['PenDigits']:
    print(f"===== {dname} =====")
    X_train, y_train = load_classification(dname, split="train", extract_path="./Multivariate_ts", load_equal_length=True)
    X_test, y_test = load_classification(dname, split="test", extract_path="./Multivariate_ts", load_equal_length=True)
    print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
    print(f"Label shape: {y_train.shape}, {y_test.shape}")

    # paddding to make the length >= 9
    X_train = np.pad(X_train, ((0, 0), (0, 0), (0, 10 - X_train.shape[2])), 'edge')
    X_test = np.pad(X_test, ((0, 0), (0, 0), (0, 10 - X_test.shape[2])), 'edge')
    print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

    for model in models:
        print(f"\t===== {model} =====")
        random.seed(SEED)
        np.random.seed(SEED)
        try:
            models[model] = train_model(models[model], X_train, y_train)
            accuracy = eval_model(models[model], X_test, y_test)
            print(f"\t{model} on {dname}: {accuracy * 100} %")
            print(f"\t{model} on {dname} object size is {sys.getsizeof(models[model])/1024} KB")
        except Exception as e:
            print(f"\t{model} on {dname} failed with error: {e}")
        models[model].reset(keep=None)
        gc.collect()

===== PenDigits =====
Train shape: (7494, 2, 8), Test shape: (3498, 2, 8)
Label shape: (7494,), (3498,)
Train shape: (7494, 2, 10), Test shape: (3498, 2, 10)
	===== MultiRocket-Hydra =====
	Function train_model took 111.6594 seconds to execute.
	Function eval_model took 2.1547 seconds to execute.
	MultiRocket-Hydra on PenDigits: 97.1412235563179 %
	MultiRocket-Hydra on PenDigits object size is 0.046875 KB
